[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/docs/semantic-search.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/docs/semantic-search.ipynb)

# Semantic Search

In this walkthrough, we'll learn how to use Pinecone for semantic search using a multilingual translation dataset.

We'll grab English sentences and search over a corpus of related sentences, aiming to find the relevant subset to our query.


Semantic search is a form of retrieval that allows you to find documents that are similar in meaning to a given query, irrespective of the words used in each query.

Semantic search is often in opposition to lexical search, where keywords are used to identify relevant documents to a given query, though it doesn't have to always be this way!

 It's super helpful for applications that require an understanding of a query's intent (such as when a user queries with a question over a corpus), or for when traditional lexical search doesn't work (such as in multimodal or multilingual applications).


To begin, let's install the following libraries:

## Installation

In [32]:
import os

from datasets import load_dataset
from pinecone import Pinecone
from tqdm import tqdm

In [33]:
!pip install -qU pinecone==8.0.0 pinecone-notebooks==0.1.1 numpy==2.0.2 datasets==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 19.4 MB/s eta 0:00:00


---

🚨 _Note: the above `pip install` is formatted for Jupyter notebooks. If running elsewhere you may need to drop the `!`._

---

## Setting up

### Get and Set the Pinecone API Key

We'll first need a free Pinecone account and API key.

This cell will help you create an account if you don't have one and then create an API key and save it in your Colab environment.

Run the cell below, and click the Pinecone Connect button to create an account or log in, and follow the prompts to generate an API key:

In [34]:
# Authenticate (only works in Colab)
try:
    from pinecone_notebooks.colab import Authenticate

    Authenticate()
except ImportError:
    # Not in Colab, skip authentication widget
    pass

Now that our key is ready, we can retrieve it from our environment and proceed:

In [35]:
# Initialize client
api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(
    # You can remove this for your own projects!
    api_key=api_key,
    source_tag="pinecone_examples:docs:semantic_search",
)

### Creating a Pinecone Index with Integrated Inference

Typically, semantic search requires three pieces: a processed data source (chunks, or records in Pinecone), an embedding model, and a vector database.

Integrated Inference allows you to specify the creation of a Pinecone index with a specific Pinecone-hosted embedding model, which makes it easy to interact with the index. To learn more about Integrated Inference, including what other models are available, take a [look here](https://docs.pinecone.io/guides/get-started/overview#integrated-embedding).


Here, we specify a starter tier index with the [llama-text-embed-v2](https://docs.pinecone.io/models/llama-text-embed-v2) embedding model. We also specify a mapping for what field in our records we will embed with this model. Then, we grab the index we just created for embedding later.

Want to instead embed a subset with multiple languages? Use the [multilingual-e5-large model](https://docs.pinecone.io/models/multilingual-e5-large) and simply specify this inplace of the previous model when creating an index.

In [37]:
index_name = "semantic-search"

existing_indexes = [index["name"] for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "multilingual-e5-large",
            "field_map": {"text": "chunk_text"}
        },
    )

index = pc.Index(name=index_name)

index.describe_index_stats()

{'dimension': 1024,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

## Creating our dataset

We're working with a small subset of a large multilingual dataset called Tatoeba. Tatoeba consists of hundreds of thousands of sentence translation pairs, and sometimes serves as a benchmark for crosslingual semantic search capabilities.

In this notebook, we're just testing semantic search, so we'll grab a subset of english sentences that include the word "park".

Why "park"? In English, park has multiple meanings which occur in different contexts. It could mean a place, such as a public park. Or, it could mean an action with a car (to park) or a place (park-ing spot). Semantic search using embedding models will naturally distinguise between these contexts, without invervention or labeling!

This is the key benefit for semantic search; a way to abstract and represent the meaning of user queries without any additional work.

And, since our embedding model is inherently multilingual, we can even do this semantic search across several languages without any additional work!

In [42]:
tatoeba = [
    {
        "translation": {
            "en": "There is a beautiful park near my house.",
            "es": "Hay un parque hermoso cerca de mi casa."
        }
    },
    {
        "translation": {
            "en": "Children are playing in the park.",
            "es": "Los niños están jugando en el parque."
        }
    },
    {
        "translation": {
            "en": "Pinecone is a vector database for semantic search.",
            "es": "Pinecone es una base de datos vectorial para búsqueda semántica."
        }
    },
]

Let's take a quick look at a few data points

In [43]:
tatoeba[0:5]

[{'translation': {'en': 'There is a beautiful park near my house.',
   'es': 'Hay un parque hermoso cerca de mi casa.'}},
 {'translation': {'en': 'Children are playing in the park.',
   'es': 'Los niños están jugando en el parque.'}},
 {'translation': {'en': 'Pinecone is a vector database for semantic search.',
   'es': 'Pinecone es una base de datos vectorial para búsqueda semántica.'}}]

In [44]:
keywords = ["park"]


def simple_keyword_filter(sentence, keywords):
    for keyword in keywords:
        if keyword.lower() in sentence.lower():
            return True
    return False


def transform_dataset_for_pinecone(dataset, use_filter=True):
    records = []

    for idx, item in enumerate(dataset):
        en_text = item["translation"]["en"]
        es_text = item["translation"]["es"]

        if use_filter and not simple_keyword_filter(en_text, keywords):
            continue

        records.append(
            {
                "id": f"en-{idx}",
                "chunk_text": en_text,
                "lang": "en"
            }
        )

        records.append(
            {
                "id": f"es-{idx}",
                "chunk_text": es_text,
                "lang": "es"
            }
        )

    return records


records = transform_dataset_for_pinecone(tatoeba, use_filter=False)

records

[{'id': 'en-0',
  'chunk_text': 'There is a beautiful park near my house.',
  'lang': 'en'},
 {'id': 'es-0',
  'chunk_text': 'Hay un parque hermoso cerca de mi casa.',
  'lang': 'es'},
 {'id': 'en-1',
  'chunk_text': 'Children are playing in the park.',
  'lang': 'en'},
 {'id': 'es-1',
  'chunk_text': 'Los niños están jugando en el parque.',
  'lang': 'es'},
 {'id': 'en-2',
  'chunk_text': 'Pinecone is a vector database for semantic search.',
  'lang': 'en'},
 {'id': 'es-2',
  'chunk_text': 'Pinecone es una base de datos vectorial para búsqueda semántica.',
  'lang': 'es'}]

## Upserting data into the Pinecone index

Here, we embed and upsert the data into Pinecone. What this means is that each record we formatted above will interact with our embedding model we specified prior, and produce a vector embedding. Then, we take these embedding batches and store them in Pinecone with the additional information we specified, which is also known as metadata.

Metadata is handy for things like filtering, like for if you stored several languages in the same index and want to return just one based on metadata. To learn more about metadata, take a [look here](https://docs.pinecone.io/guides/index-data/indexing-overview#metadata).

We specify and create a namespace called "english-sentences", which is a higher level unit of organization when interacting with Pinecone.

Querying on namespaces performs a sort of broad filter to only records that exist in that namespace, which has the nice effect of speeding up searches too.

To learn more about namespaces, [look here](https://docs.pinecone.io/guides/index-data/indexing-overview#namespaces)


In [48]:
batch_size = 96
namespace = "english-sentences"


# We upsert in batches of 96 to avoid hitting the embedding model's rate limit.
# Libraries like backoff can be used here to handler large embedding jobs.

for start in tqdm(range(0, len(records), batch_size), "Upserting records batch: "):
    batch = records[start : start + batch_size]

    embeddings = pc.inference.embed(
        model="multilingual-e5-large",
        inputs=[r["chunk_text"] for r in batch],
        parameters={
            "input_type": "passage",
            "truncate": "END"
        }
    )

    vectors = [
        {
            "id": r["id"],
            "values": e["values"],
            "metadata": {
                "chunk_text": r["chunk_text"],
                "lang": r["lang"]
            }
        }
        for r, e in zip(batch, embeddings)
    ]

    index.upsert(
        vectors=vectors,
        namespace=namespace
    )

Upserting records batch: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


## Making Queries

Now that our index is populated we can begin making queries.

The tricky part about querying with semantic search is that we'd normally need to involve an embedding model here again too!

But with Pinecone's Integrated Inference, we can just invoke our index we created and send the text we want to search with there. Specifically,
the search query is vectorized using the same embedding model we specified prior, and we use this vector to find all closest vectors in the database to it to return.

Neat!

Our goal here is to write a query sentence that uses one form of the word park, and find sentences that use park in a semantically similar manner. So, let's try this:


In [50]:
search_query = "I want to go to the park and relax"

query_embedding = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[search_query],
    parameters={
        "input_type": "query",
        "truncate": "END"
    }
)[0]["values"]

results = index.query(
    namespace=namespace,
    vector=query_embedding,
    top_k=10,
    include_metadata=True
)

for match in results["matches"]:
    print(
        f"Sentence: {match['metadata']['chunk_text']} "
        f"Semantic Similarity Score: {match['score']}\n"
    )

Sentence: There is a beautiful park near my house. Semantic Similarity Score: 0.801927686

Sentence: Hay un parque hermoso cerca de mi casa. Semantic Similarity Score: 0.785677969

Sentence: Children are playing in the park. Semantic Similarity Score: 0.779759705

Sentence: Los niños están jugando en el parque. Semantic Similarity Score: 0.766468

Sentence: Pinecone es una base de datos vectorial para búsqueda semántica. Semantic Similarity Score: 0.724127531

Sentence: Pinecone is a vector database for semantic search. Semantic Similarity Score: 0.71462363



And now, let's use the other meaning of the word, park!

In [52]:
search_query = "I need a place to park"

query_embedding = pc.inference.embed(
    model="multilingual-e5-large",
    inputs=[search_query],
    parameters={
        "input_type": "query",
        "truncate": "END"
    }
)[0]["values"]

results = index.query(
    namespace=namespace,
    vector=query_embedding,
    top_k=10,
    include_metadata=True
)

for match in results["matches"]:
    print(
        f"Sentence: {match['metadata']['chunk_text']} "
        f"Semantic Similarity Score: {match['score']}\n"
    )

Sentence: There is a beautiful park near my house. Semantic Similarity Score: 0.781433582

Sentence: Hay un parque hermoso cerca de mi casa. Semantic Similarity Score: 0.761459589

Sentence: Pinecone es una base de datos vectorial para búsqueda semántica. Semantic Similarity Score: 0.733837426

Sentence: Pinecone is a vector database for semantic search. Semantic Similarity Score: 0.730864346

Sentence: Children are playing in the park. Semantic Similarity Score: 0.729175329

Sentence: Los niños están jugando en el parque. Semantic Similarity Score: 0.720616937



## Wait, how is this working?

When performing semantic search with Pinecone's vector database, you are asking the following question: Given this query vector, what are the closest vectors to it in the database?

Because of the way embedding models are trained, this closeness in vector space corresponds to similarity in meaning. The exact metric used for our implementation is cosine similarity, which is simply the angle between the input vector and a document vector. For small amounts of vectors, this task is trivial, but what happens when you have hundreds of thousands, millions or even billions? And what about query latency?

The magic of Pinecone's vector database is advanced algorithms that can quickly index and do this search on billion-scale vectors effectively!

## Demo Cleanup

You can go ahead and ask more queries above. When you're done, delete the index to save resources.

Congrats, you've just implemented semantic search with Pinecone!


In [ ]:
pc.delete_index(name=index_name)

---